# Checkpoint B3: Extraction Agent — Gọi Gemini API trích xuất điều khoản

Checkpoint này triển khai hàm `extract_contract_terms()` gọi Gemini API để trích xuất điều khoản hợp đồng theo JSON schema.

**Điều kiện đạt:**
- [x] System prompt định nghĩa rõ vai trò + ràng buộc đầu ra
- [x] User prompt template chèn nội dung hợp đồng + JSON schema
- [x] Hàm gọi Gemini API với `google-generativeai` library
- [x] Helper `safe_parse_json()` xử lý response an toàn
- [x] Trả về dict JSON hoặc error message

**Tiếp theo:** Sau khi pass, chuyển sang Checkpoint B4 (self-check).

In [ ]:
import json
import re

# ============================================================
# Cài đặt thư viện (chạy trên Colab)
# ============================================================
# !pip install -q google-generativeai

import google.generativeai as genai

# ============================================================
# System prompt — Định nghĩa vai trò và ràng buộc đầu ra
# ============================================================
SYSTEM_PROMPT = """Bạn là trợ lý rà soát điều khoản hợp đồng cho đội vận hành viễn thông.

NHIỆM VỤ:
Đọc hợp đồng dưới đây đã được nhận dạng quang học (OCR). Trích xuất các điều khoản quan trọng theo JSON schema được cung cấp.

ĐẦU RA:
Trả về JSON với cấu trúc chính xác theo schema. Mọi trường trích xuất được phải kèm source_evidence trích dẫn nguyên văn từ hợp đồng.

RANH GIỚI:
- Chỉ kết luận dựa trên nội dung có trong hợp đồng
- Không suy đoán khi thiếu dữ liệu
- Không bổ sung thông tin không có trong hợp đồng

QUY TẮC AN TOÀN:
- Thiếu căn cứ → ghi needs_human_review=true
- Mâu thuẫn giữa các điều khoản → ghi needs_human_review=true + thêm vào red_flags[]
- Confidence phản ánh mức chắc chắn dựa trên căn cứ nguồn, không phải cảm tính
- Mọi trường phải kèm source_evidence trích dẫn nguyên văn

RÀNG BUỘC:
- source_evidence phải chứa: field, quote (trích nguyên văn), section (điều khoản số mấy)
- Nếu trường không tìm thấy trong hợp đồng → thêm vào missing_fields[], không để trống hoặc tự tạo
- penalty_amount phải là chuỗi chứa số tiền và đơn vị
"""

# ============================================================
# User prompt template — Chèn nội dung hợp đồng vào
# ============================================================
USER_PROMPT_TEMPLATE = """Hợp đồng cần trích xuất:

---
{contract_content}
---

JSON schema đầu ra (các trường bắt buộc):
- contract_id: string (mã hợp đồng)
- effective_date: string (YYYY-MM-DD, null nếu không tìm thấy)
- expiry_date: string (YYYY-MM-DD, null nếu không tìm thấy)
- penalty_clause: string (tóm tắt điều khoản phạt)
- penalty_amount: string (số tiền/mức phạt chi tiết)
- source_evidence: array của {{field, quote, section}}
- confidence: number 0.0-1.0
- needs_human_review: boolean
- red_flags: array of string
- missing_fields: array of string
- extraction_notes: string

Yêu cầu: Trích xuất điều khoản từ hợp đồng trên theo JSON schema.
Đảm bảo mọi trường có source_evidence.
Đánh giá confidence dựa trên số lượng căn cứ nguồn.
Nếu phát hiện cờ đỏ, thêm vào red_flags[].
Nếu thiếu trường hoặc không chắc chắn, bật needs_human_review=true.

TRẢ VỀ JSON HỢP LỆ, KHÔNG CÓ MARKDOWN CODE BLOCKS."""

# ============================================================
# Helper: Parse JSON an toàn từ LLM response
# ============================================================
def safe_parse_json(text: str) -> dict | None:
    """Parse JSON từ LLM response, xử lý các trường hợp phổ biến:
    - JSON bọc trong ```json ... ```
    - JSON có trailing commas
    - Text thừa trước/sau JSON

    Args:
        text: Chuỗi response từ LLM

    Returns:
        Dict nếu parse thành công, None nếu thất bại
    """
    # Trích xuất JSON từ markdown code block nếu có
    cleaned = text.strip()
    markdown_match = re.search(r'```(?:json)?\s*\n(.*?)\n\s*```', cleaned, re.DOTALL | re.IGNORECASE)
    if markdown_match:
        cleaned = markdown_match.group(1).strip()
    else:
        # Nếu không có code block bọc ngoài nhưng vẫn bắt đầu bằng ``` (trường hợp bị cắt cụt)
        if cleaned.startswith("```"):
            lines = cleaned.split("\n")
            lines = lines[1:]  # Bỏ dòng ```json đầu
            if lines and lines[-1].strip() == "```":
                lines = lines[:-1]  # Bỏ dòng ``` cuối
            cleaned = "\n".join(lines).strip()

    # Thử parse trực tiếp
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Nếu vẫn lỗi, tìm JSON object trong text (tìm { ... } outermost bằng cách đếm ngoặc nhọn)
    first_brace = cleaned.find('{')
    if first_brace != -1:
        brace_count = 0
        in_string = False
        escape = False
        for i in range(first_brace, len(cleaned)):
            char = cleaned[i]
            if escape:
                escape = False
                continue
            if char == '\\':
                escape = True
                continue
            if char == '"':
                in_string = not in_string
                continue
            if not in_string:
                if char == '{':
                    brace_count += 1
                elif char == '}':
                    brace_count -= 1
                    if brace_count == 0:
                        potential_json = cleaned[first_brace:i+1]
                        try:
                            return json.loads(potential_json)
                        except json.JSONDecodeError:
                            pass
                        break
    return None


# ============================================================
# Hàm chính: Gọi Gemini API trích xuất điều khoản
# ============================================================
def extract_contract_terms(contract_text: str, api_key: str) -> dict:
    """Gọi Gemini API để trích xuất điều khoản từ hợp đồng.

    Workflow:
    1. Cấu hình API key
    2. Tạo model (dùng gemini-2.5-flash theo chuẩn May 2026)
    3. Build prompt từ template
    4. Gọi API và nhận response
    5. Parse JSON bằng safe_parse_json()
    6. Trả về dict hoặc error

    Args:
        contract_text: Nội dung đầy đủ hợp đồng
        api_key: Google AI API key (lấy từ AI Studio)

    Returns:
        Dict chứa kết quả trích xuất hoặc {"error": "..."}
    """
    try:
        # Bước 1: Cấu hình API
        genai.configure(api_key=api_key)

        # Bước 2: Tạo model — dùng gemini-flash-latest theo tiêu chuẩn 2026
        model = genai.GenerativeModel(
            model_name="gemini-flash-latest",
            system_instruction=SYSTEM_PROMPT,
        )

        # Bước 3: Build user prompt
        user_prompt = USER_PROMPT_TEMPLATE.format(contract_content=contract_text)

        # Bước 4: Gọi API
        response = model.generate_content(user_prompt)

        # Bước 5: Parse JSON response
        raw_text = response.text
        result = safe_parse_json(raw_text)

        if result is None:
            return {
                "error": "Không parse được JSON từ API response",
                "raw_response": raw_text[:500],
            }

        # Bước 6: Trả về kết quả
        return result

    except Exception as e:
        return {"error": f"Lỗi khi gọi API: {str(e)}"}


# ============================================================
# Ví dụ JSON output kỳ vọng (dùng để tham khảo)
# ============================================================
EXAMPLE_OUTPUT = {
    "contract_id": "contract-001",
    "effective_date": "2026-01-01",
    "expiry_date": "2026-12-31",
    "penalty_clause": "Phạt 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất, tối đa 10% giá trị hợp đồng hàng quý",
    "penalty_amount": "1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất",
    "source_evidence": [
        {"field": "effective_date", "quote": "Hợp đồng có hiệu lực kể từ ngày 01 tháng 01 năm 2026", "section": "Điều 2"},
        {"field": "expiry_date", "quote": "đến ngày 31 tháng 12 năm 2026", "section": "Điều 2"},
        {"field": "penalty_clause", "quote": "Phạt 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất", "section": "Điều 4"},
    ],
    "confidence": 0.92,
    "needs_human_review": False,
    "red_flags": [],
    "missing_fields": [],
    "extraction_notes": "Hợp đồng đầy đủ, rõ ràng. Không phát hiện cờ đỏ.",
}
print("Ví dụ JSON output kỳ vọng:")
print(json.dumps(EXAMPLE_OUTPUT, indent=2, ensure_ascii=False))

Ví dụ JSON output kỳ vọng:
{
  "contract_id": "contract-001",
  "effective_date": "2026-01-01",
  "expiry_date": "2026-12-31",
  "penalty_clause": "Phạt 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất, tối đa 10% giá trị hợp đồng hàng quý",
  "penalty_amount": "1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất",
  "source_evidence": [
    {
      "field": "effective_date",
      "quote": "Hợp đồng có hiệu lực kể từ ngày 01 tháng 01 năm 2026",
      "section": "Điều 2"
    },
    {
      "field": "expiry_date",
      "quote": "đến ngày 31 tháng 12 năm 2026",
      "section": "Điều 2"
    },
    {
      "field": "penalty_clause",
      "quote": "Phạt 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất",
      "section": "Điều 4"
    }
  ],
  "confidence": 0.92,
  "needs_human_review": false,
  "red_flags": [],
  "missing_fields": [],
  "extraction_notes": "Hợp đồng đầy đủ, rõ ràng. Không phát hiện cờ đỏ."
}


In [ ]:
# ============================================================
# Test extraction trên contract-001.docx (yêu cầu API key)
# ============================================================
# Chỉ chạy cell này khi đã có API key từ Google AI Studio
# https://aistudio.google.com/app/apikey

import os
try:
    from docx import Document as DocxDocument
    HAS_DOCX = True
except ImportError:
    HAS_DOCX = False

# Tự động phát hiện PROJECT_ROOT dựa trên vị trí file notebook hoặc thư mục chạy hiện tại
possible_roots = [
    "../../outputs/contract-term-extractor",
    "../outputs/contract-term-extractor",
    "outputs/contract-term-extractor",
    "03-practice/day-02/session-03-ai-agent-design/outputs/contract-term-extractor"
]

PROJECT_ROOT = None
for root in possible_roots:
    if os.path.exists(root):
        PROJECT_ROOT = root
        break

if not PROJECT_ROOT:
    PROJECT_ROOT = "../../outputs/contract-term-extractor"

contracts_dir = f"{PROJECT_ROOT}/data/contracts"
contracts = {}

if HAS_DOCX and os.path.exists(contracts_dir):
    for fname in os.listdir(contracts_dir):
        if fname.endswith(".docx"):
            fpath = os.path.join(contracts_dir, fname)
            try:
                doc = DocxDocument(fpath)
                contracts[fname] = "\n".join(p.text for p in doc.paragraphs)
            except Exception as e:
                print(f"Lỗi đọc {fname}: {e}")

# Cách 1: Nhập API key trực tiếp hoặc tự động lấy từ environment
API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY") or "your-api-key-here"

# Cách 2: Dùng Colab secret (khuyến nghị)
# from google.colab import userdata
# API_KEY = userdata.get('GOOGLE_API_KEY')

# --- Chạy extraction (chỉ thực hiện khi có API key hợp lệ) ---
placeholders = ["your-api-key-here", "YOUR_GEMINI_API_KEY_HERE", "YOUR_API_KEY_HERE", ""]
is_valid_key = API_KEY and API_KEY.strip() not in placeholders and not API_KEY.startswith("YOUR_")

if is_valid_key:
    if contracts:
        result = extract_contract_terms(contracts.get("contract-001.docx", ""), API_KEY)
        print("Kết quả trích xuất contract-001 (gọi API thật):")
        print(json.dumps(result, indent=2, ensure_ascii=False))
    else:
        print("Chưa load được contracts từ data/contracts hoặc thiếu python-docx")
else:
    print("Bỏ qua gọi API thật vì không cấu hình API_KEY hợp lệ.")

# ============================================================
# Demo: Chạy với mock data (không cần API key)
# ============================================================
print("\n=== Demo với mock data (không gọi API thật) ===\n")

# Mock kết quả trích xuất cho contract-001
mock_result = {
    "contract_id": "contract-001",
    "effective_date": "2026-01-01",
    "expiry_date": "2026-12-31",
    "penalty_clause": "Phạt 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất, tối đa 10%",
    "penalty_amount": "1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất",
    "source_evidence": [
        {"field": "effective_date", "quote": "Hợp đồng có hiệu lực kể từ ngày 01 tháng 01 năm 2026", "section": "Điều 2"},
        {"field": "expiry_date", "quote": "đến ngày 31 tháng 12 năm 2026", "section": "Điều 2"},
        {"field": "penalty_clause", "quote": "Phạt 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất", "section": "Điều 4"},
    ],
    "confidence": 0.92,
    "needs_human_review": False,
    "red_flags": [],
    "missing_fields": [],
    "extraction_notes": "Hợp đồng đầy đủ, rõ ràng.",
}

print("Kết quả trích xuất contract-001:")
print(json.dumps(mock_result, indent=2, ensure_ascii=False))

print("\n--- Tóm tắt ---")
print(f"Contract: {mock_result['contract_id']}")
print(f"Thời hạn: {mock_result['effective_date']} → {mock_result['expiry_date']}")
print(f"Confidence: {mock_result['confidence']}")
print(f"Needs HITL: {mock_result['needs_human_review']}")
print(f"Red flags: {len(mock_result['red_flags'])}")
print(f"Source evidence: {len(mock_result['source_evidence'])} items")

print("\n--- Bước tiếp theo ---")
print("1. Thay API_KEY bằng key thật để gọi Gemini API")
print("2. Chạy extraction cho contract-002.docx và contract-003-risky.docx")
print("3. So sánh kết quả — đặc biệt missing_fields và red_flags")

Kết quả trích xuất contract-001 (gọi API thật):
{
  "contract_id": "HD-DV-2026-001",
  "effective_date": "2026-01-01",
  "expiry_date": "2026-12-31",
  "penalty_clause": "Điều khoản phạt áp dụng cho Bên A nếu không đảm bảo cam kết sẵn sàng 99.5% (phạt 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất, tối đa 10% giá trị hợp đồng hàng quý). Điều khoản phạt áp dụng cho Bên B nếu thanh toán chậm (phạt 0.05% giá trị chưa thanh toán cho mỗi ngày chậm, có thể bị tạm ngưng dịch vụ sau 30 ngày chậm).",
  "penalty_amount": "Đối với Bên A: 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất, tối đa 10% giá trị hợp đồng hàng quý. Đối với Bên B: 0.05% giá trị chưa thanh toán cho mỗi ngày chậm.",
  "source_evidence": [
    {
      "field": "contract_id",
      "quote": "Số: HD-DV-2026-001",
      "section": "Header"
    },
    {
      "field": "effective_date",
      "quote": "Hợp đồng có hiệu lực kể từ ngày 01 tháng 01 năm 2026.",
      "section": "ĐIỀU 2: THỜI HẠN HỢP ĐỒNG"
    },